# §12.5.5 — 기울기 노름 분포로 임계값 정하기

> 딥러닝 교재 · 3부 12장 5절 5항 (🐍)
> 선행: §12.5.1(노름 클리핑) · §12.5.4(유효 학습률과 두 체제) · §12.5.7(발동 빈도 진단)

## 이 노트북이 답하는 질문

1. **기울기 노름의 분포는 어떤 모양인가?** 좁은 본체와 몇 자릿수를 뛰는 꼬리를 확인한다.
2. **임계값은 어디에 두어야 하는가?** 임계값별 발동 빈도와 최종 성능으로 답한다.
3. **클리핑의 이득은 평균인가 분산인가?** 같은 설정을 씨앗만 바꿔 반복해 실패율을 센다.

**예상 실행 시간** CPU 약 2분 (`FAST = True`이면 약 1분).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제·모델·그리고 두 줄의 클리핑

과제는 §12.3.5와 같은 지연 회상 $y=\operatorname{sign}(x_{T-\Delta})$이다. 다만 이번에는 폭발이
**실제로 일어나는 지형**이 필요하므로, 스펙트럼 반경을 약간 초임계($\rho_0=1.15$)로 두고,
배치를 작게($B=16$) 하고, 학습률을 일부러 절벽이 밟히는 크기(0.013)로 잡는다.

옵티마이저는 **모멘텀 SGD**를 쓴다. Adam류는 좌표별 정규화가 내장되어 있어 폭발 스텝의
피해가 가려지고, 그만큼 클리핑의 효과도 보이지 않게 되기 때문이다(§12.5.4의 유효 학습률
논의가 그대로 적용되는 것은 SGD 계열이다).

클리핑 자체는 본문 §12.5.1의 마진 리스팅 그대로, 두 줄이 전부다.

In [ ]:
M = 32          # 상태 차원
DELTA = 10      # 의존 거리
T_SEQ = DELTA + 5
RHO0 = 1.15     # 초기 스펙트럼 반경 — 약간 초임계로 절벽을 만든다
LR = 0.013      # 학습률 (모멘텀 SGD) — 절벽이 실제로 밟히는 크기
MOM = 0.9
B_TR = 16       # 작은 배치 — 꼬리를 살린다
STEPS = 500 if FAST else 1000
EVAL_EVERY = 25

def make_W(rho, rn, m=M):
    W = rn.standard_normal((m, m)) / np.sqrt(m)
    W *= rho / np.abs(np.linalg.eigvals(W)).max()
    return W

def init_params(seed):
    rn = np.random.default_rng(seed)
    Wh = make_W(RHO0, rn)
    Wx = rn.standard_normal((1, M))
    b = np.zeros(M)
    U = rn.standard_normal(M) / np.sqrt(M)
    c_out = np.zeros(1)
    return [Wh, Wx, b, U, c_out]

def forward_backward(params, X, y01):
    # 완전한 BPTT — 손실·정확도·기울기 리스트를 돌려준다
    Wh, Wx, b, U, c_out = params
    B, T_ = X.shape
    H = np.zeros((B, T_ + 1, M)); Z = np.zeros((B, T_, M))
    for t in range(T_):
        Z[:, t] = H[:, t] @ Wh + X[:, t:t+1] @ Wx + b
        H[:, t+1] = np.tanh(Z[:, t])
    logit = H[:, T_] @ U + c_out
    p = np.where(logit >= 0, 1/(1+np.exp(-logit)), np.exp(logit)/(1+np.exp(logit))).ravel()
    eps = 1e-12
    loss = -np.mean(y01*np.log(p+eps) + (1-y01)*np.log(1-p+eps))
    acc = np.mean((logit.ravel() > 0) == (y01 > 0.5))
    dlogit = (p - y01) / B
    gU = H[:, T_].T @ dlogit; gc = np.array([dlogit.sum()])
    delta_b = np.outer(dlogit, U)
    gWh = np.zeros_like(Wh); gWx = np.zeros_like(Wx); gb = np.zeros_like(b)
    for t in range(T_ - 1, -1, -1):
        dz = delta_b * (1 - np.tanh(Z[:, t])**2)
        gWh += H[:, t].T @ dz; gWx += X[:, t:t+1].T @ dz; gb += dz.sum(0)
        delta_b = dz @ Wh.T
    return loss, acc, [gWh, gWx, gb, gU, gc]

def gnorm(grads):
    return np.sqrt(sum((g**2).sum() for g in grads))

def clip_norm(grads, c):
    # §12.5.1 마진 리스팅의 두 줄 — 방향은 보존, 크기만 제한
    n = gnorm(grads)
    if c is not None and n > c:
        grads = [g * (c / n) for g in grads]
    return grads, n

def eval_acc(params, n=1500):
    Wh, Wx, b, U, c_out = params
    rn = np.random.default_rng(SEED + 1)
    X = rn.standard_normal((n, T_SEQ))
    y = X[:, T_SEQ - 1 - DELTA] > 0
    h = np.zeros((n, M))
    for t in range(T_SEQ):
        h = np.tanh(h @ Wh + X[:, t:t+1] @ Wx + b)
    return np.mean(((h @ U + c_out) > 0) == y)

def train(seed, theta_c=None, steps=STEPS, record_curve=False):
    # theta_c=None 이면 클리핑 없음. 노름 기록·발동 비율·평가 곡선을 돌려준다.
    params = init_params(seed)
    vel = [np.zeros_like(p) for p in params]
    rb = np.random.default_rng(7000 + seed)
    norms = np.zeros(steps); fired = 0
    curve_t, curve_a = [], []
    for t in range(steps):
        X = rb.standard_normal((B_TR, T_SEQ))
        y01 = (X[:, T_SEQ - 1 - DELTA] > 0).astype(float)
        loss, acc, grads = forward_backward(params, X, y01)
        grads, n = clip_norm(grads, theta_c)
        norms[t] = n
        if theta_c is not None and n > theta_c:
            fired += 1
        for p_, g_, v_ in zip(params, grads, vel):
            v_[:] = MOM * v_ + g_
            p_ -= LR * v_
        if record_curve and (t % EVAL_EVERY == 0 or t == steps - 1):
            curve_t.append(t); curve_a.append(eval_acc(params, n=600))
    return {'norms': norms, 'fired': fired / steps, 'final': eval_acc(params),
            'curve': (np.array(curve_t), np.array(curve_a))}

print(f"설정: m={M}, Δ={DELTA}, T={T_SEQ}, ρ0={RHO0}, lr={LR}, B={B_TR}, steps={STEPS}")

---
## 2. 진단 실행 — 클리핑 없이 노름을 기록한다

§12.5.7의 순서 그대로다. 임계값을 고르기 **전에**, 클리핑 없는 실행에서 스텝별 기울기
노름을 기록해 분포부터 본다. 한 가지 실무적 주의: 폭발이 일어난 뒤의 노름은 이미 병든
지형의 기록이고, 본체 자체도 학습이 진행되며 위로 이동한다. 그래서 본체의 통계는
**첫 폭발 이전, 학습 초반의 구간**(여기서는 처음 200스텝)에서 잰다 — 실무의 짧은
파일럿 실행에 해당한다.

In [ ]:
DIAG_SEED = 301
diag = train(seed=DIAG_SEED, theta_c=None, record_curve=True)
ns = diag['norms']

SPIKE_LVL = 10.0                       # 이 수준을 넘으면 폭발 스텝으로 본다
first_spike = int(np.argmax(ns > SPIKE_LVL)) if (ns > SPIKE_LVL).any() else len(ns)
healthy = ns[:min(max(first_spike, 100), 200)]   # 첫 폭발 이전·초반의 파일럿 구간
pct = {q: np.percentile(healthy, q) for q in [50, 90, 95, 99]}
print(f"첫 폭발(노름>{SPIKE_LVL:g})은 스텝 {first_spike}에서 | 폭발 스텝 수 {(ns > SPIKE_LVL).sum()}")
print(f"건강 구간 노름 분포:  중앙값 {pct[50]:.2f} | p90 {pct[90]:.2f} | "
      f"p95 {pct[95]:.2f} | p99 {pct[99]:.2f}")
print(f"전 구간 최대 {ns.max():.0f}  ← 본체 중앙값의 {ns.max()/pct[50]:.0f}배")
print(f"참고: θ_c=1.5는 이 구간 분포의 p{100*np.mean(healthy <= 1.5):.0f} 지점")

본체(건강 구간)의 중앙값과 전 구간 최댓값이 자릿수 단위로 떨어져 있다 — 분포는 "좁은
본체 + 긴 꼬리"다. 임계값의 후보는 본체 분포의 **상위 몇 퍼센트 지점**(p95–p99 근방)이다.

---
## 3. 임계값 훑기 — 발동 빈도와 최종 성능

임계값을 본체 깊숙한 곳부터 꼬리 바깥까지 로그 격자로 훑는다. 각 임계값마다 씨앗을
바꿔 여러 번 학습하고, 발동 빈도(전체 스텝 중 클리핑이 일어난 비율)와 최종 정확도를
기록한다.

In [ ]:
THETAS = [0.05, 0.2, 0.5, 1.5, 5.0, 15.0, None]   # None = 클리핑 없음
N_SEEDS = 3 if FAST else 5
sweep = {}
for th in THETAS:
    runs = [train(seed=100 + s, theta_c=th) for s in range(N_SEEDS)]
    sweep[th] = {'fired': np.mean([r['fired'] for r in runs]),
                 'finals': np.array([r['final'] for r in runs])}
    nm = '없음' if th is None else f'{th:g}'
    print(f"θ_c={nm:>5}:  발동 {sweep[th]['fired']*100:5.1f}%  "
          f"정확도 중앙값 {np.median(sweep[th]['finals']):.3f}  "
          f"(최저 {sweep[th]['finals'].min():.3f} / 최고 {sweep[th]['finals'].max():.3f})")

---
## 4. 안정성 — 같은 설정, 씨앗만 바꿔 반복

파일럿 구간 분포의 상위 십 퍼센트 부근에 해당하는 임계값 하나($\theta_c=1.5\approx$ p90)를
골라, 클리핑 유무만 다른 두 조건을 씨앗을 바꿔 반복한다. 관심사는 평균 성능이 아니라
**실패한 실행의 빈도**다.

In [ ]:
THETA_STAR = 1.5
N_REP = 4 if FAST else 10
curves_off = [train(seed=300 + s, theta_c=None, record_curve=True) for s in range(N_REP)]
curves_on  = [train(seed=300 + s, theta_c=THETA_STAR, record_curve=True) for s in range(N_REP)]

FAIL_LVL = 0.8
fail_off = sum(1 for r in curves_off if r['final'] < FAIL_LVL)
fail_on  = sum(1 for r in curves_on if r['final'] < FAIL_LVL)
ok_off = [r['final'] for r in curves_off if r['final'] >= FAIL_LVL]
print(f"클리핑 없음:  {N_REP}런 중 {fail_off}런 실패(정확도<{FAIL_LVL})"
      f"  | 생존 런 중앙값 {np.median(ok_off) if ok_off else float('nan'):.3f}")
print(f"θ_c={THETA_STAR}:      {N_REP}런 중 {fail_on}런 실패"
      f"           | 전체 중앙값 {np.median([r['final'] for r in curves_on]):.3f}")

---
## 5. 교재 그림 — fig_12_5_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 노름 분포 — 로그 축의 본체와 꼬리
ax = axes[0]
bins = np.logspace(np.log10(max(ns.min(), 1e-2)), np.log10(ns.max() * 1.2), 40)
ax.hist(ns, bins=bins, color=CB[2], alpha=0.85)
ax.set_xscale('log')
ax.axvline(pct[50], color='k', lw=0.9, ls=':')
ax.axvline(pct[99], color=CB[4], lw=1.2, ls='--')
ymax = ax.get_ylim()[1]
ax.text(pct[50]*0.92, ymax*0.93, lab('본체 중앙값', 'median'), fontsize=8, ha='right')
ax.text(pct[99]*1.08, ymax*0.93, lab('본체 p99', 'p99'), color=CB[4], fontsize=8)
ax.annotate(lab('꼬리(폭발 스텝)', 'tail (spikes)'), xy=(ns.max()*0.6, 2),
            xytext=(ns.max()*0.05, ymax*0.5), color=CB[4], fontsize=9,
            arrowprops=dict(arrowstyle='->', color=CB[4]))
ax.set_xlabel(lab('기울기 노름 $\\|g\\|$ (로그 축)', 'gradient norm (log)'))
ax.set_ylabel(lab('스텝 수', 'count'))
ax.set_title(lab('(a) 좁은 본체와 긴 꼬리 — 클리핑 없는 실행', '(a) body and tail'), fontsize=10)

# (b) 노름 시계열과 폭발 스텝
ax = axes[1]
steps_ax = np.arange(len(ns))
ax.semilogy(steps_ax, ns, '-', color=CB[0], lw=0.5, alpha=0.75)
spikes = steps_ax[ns > SPIKE_LVL]
ax.semilogy(spikes, ns[spikes], 'o', color=CB[4], ms=3.5,
            label=lab(f'폭발 스텝 (>{SPIKE_LVL:g})', 'spikes'))
ax.axhline(THETA_STAR, color=CB[5], lw=1.1, ls='--', label=f'$\\theta_c={THETA_STAR}$')
ax.axhline(0.05, color=CB[1], lw=1.1, ls='--', label='$\\theta_c=0.05$')
ax.set_xlabel(lab('학습 걸음', 'step'))
ax.set_ylabel('$\\|g\\|$')
ax.set_title(lab('(b) 노름의 시계열 — 클리핑 없음', '(b) norm time series'), fontsize=10)
ax.legend(fontsize=8, loc='upper left')

# (c) 임계값별 발동 빈도와 최종 성능
ax = axes[2]
xs = [th for th in THETAS if th is not None]
med = [np.median(sweep[th]['finals']) for th in xs]
lo = [sweep[th]['finals'].min() for th in xs]
hi = [sweep[th]['finals'].max() for th in xs]
ax.plot(xs, med, 'o-', color=CB[5], ms=5, label=lab('최종 정확도(중앙값)', 'final acc (median)'))
ax.fill_between(xs, lo, hi, color=CB[5], alpha=0.18, label=lab('씨앗별 최저–최고', 'min–max'))
ax.axhline(np.median(sweep[None]['finals']), color=CB[0], lw=0.9, ls=':',
           label=lab('클리핑 없음(중앙값)', 'no clip'))
ax.axvline(pct[50], color='k', lw=0.8, ls=':')
ax.axvline(pct[99], color=CB[4], lw=0.8, ls='--')
ax.text(pct[50]*0.9, 0.47, lab('본체 중앙값', 'median'), fontsize=7, rotation=90, va='bottom', ha='right')
ax.text(pct[99]*1.1, 0.47, lab('본체 p99', 'p99'), color=CB[4], fontsize=7, rotation=90, va='bottom')
ax.set_xscale('log')
ax.set_xlabel(lab('임계값 $\\theta_c$ (로그 축)', 'threshold (log)'))
ax.set_ylabel(lab('시험 정확도', 'test acc'), color=CB[5])
ax2 = ax.twinx()
ax2.plot(xs, [sweep[th]['fired']*100 for th in xs], 's--', color=CB[1], ms=4, lw=1)
ax2.set_ylabel(lab('발동 빈도 (%)', 'trigger freq (%)'), color=CB[1])
ax2.tick_params(axis='y', colors=CB[1]); ax2.grid(False)
ax.set_title(lab('(c) 본체를 파고들면 손해, 꼬리 바깥이면 무용', '(c) threshold sweep'), fontsize=10)
ax.legend(fontsize=7, loc='lower center')

# (d) 클리핑 유무에 따른 학습 곡선
ax = axes[3]
for i, r in enumerate(curves_off):
    t_, a_ = r['curve']
    ax.plot(t_, a_, '-', color=CB[4], lw=0.9, alpha=0.7,
            label=lab('클리핑 없음', 'no clip') if i == 0 else None)
for i, r in enumerate(curves_on):
    t_, a_ = r['curve']
    ax.plot(t_, a_, '-', color=CB[5], lw=0.9, alpha=0.7,
            label=f'$\\theta_c={THETA_STAR}$' if i == 0 else None)
ax.axhline(0.5, color='k', lw=0.6, ls=':')
ax.set_xlabel(lab('학습 걸음', 'step'))
ax.set_ylabel(lab('시험 정확도', 'test acc'))
ax.set_title(lab(f'(d) 씨앗 {N_REP}개 반복 — 이득은 분산에 있다', '(d) stability over seeds'), fontsize=10)
ax.legend(fontsize=8, loc='lower right')

save_book_fig(fig, 'fig_12_5_5')
plt.show()

> ### 읽는 법
>
> (a) 노름 분포는 좁은 본체와, 본체 중앙값에서 자릿수 단위로 뛰는 긴 꼬리로 이루어진다.
> 꼬리의 정체는 (b)의 산발적 폭발 스텝 — §12.3.2의 초임계 스펙트럼이 만든 절벽 지형을
> 밟는 순간들이다.
> (c) 임계값이 본체를 파고들면(발동이 사실상 100%) 유효 걸음이 $\eta\theta_c$로 눌린
> 정규화 SGD가 되어(§12.5.4의 (i) 체제) 같은 예산 안에서 학습이 느려지고, 꼬리 바깥으로
> 나가면 폭발 스텝이 통과해 클리핑 없음과 같아진다. 최선은 그 사이, 본체 상단
> (p90–p95 근방)이다.
> (d) 그 임계값의 첫 번째 이득은 **실패 빈도**다. 클리핑 없는 실행의 상당수는 폭발 후
> 회복하지 못하고, 클리핑을 켜면 실패가 사라진다. 생존한 실행들도 잔폭발의 비용을
> 치러 중앙값이 한 계단 낮다 — 곡선 다발의 폭(분산)을 접는 것이 클리핑의 일이다.
>
> 한 가지 정직한 주석: 이 실험의 최적 임계값에서도 발동 빈도는 수십 퍼센트에 이른다.
> 학습률을 일부러 절벽이 밟히는 크기로 잡았기 때문인데, **발동이 이렇게 잦다는 것
> 자체가 §12.5.7이 말하는 "학습률이 지형에 비해 크다"는 신호다.** 실제로 `LR`을 절반으로
> 내리면 폭발 자체가 드물어져 클리핑 없이도 실패가 사라진다 — 클리핑은 증상 완화이고
> 원인 처방은 따로 있다는 §12.5.6의 문장이 손잡이 하나로 확인된다.

---
## 6. 자기 점검

1. (c)에서 가장 작은 임계값 두 개의 발동 빈도는 100%다. §12.5.4의 (i) 체제에서 $(\eta,\theta_c)$가 곱으로만 작용함을 이용해, $\theta_c=0.05$ 결과를 재현하는 다른 $(\eta,\theta_c)$ 조합을 하나 제시하고 실행으로 확인하라.
2. (b)의 폭발 스텝과 (d)의 곡선이 꺾이는 지점을 짝지어 보라. 회복하는 런과 못 하는 런의 차이는 무엇인가?
3. `LR = 0.0065`로 바꿔 4절을 다시 실행해 보라. 실패율과 발동 빈도가 어떻게 변하는가? "발동이 지속적으로 잦다면 학습률이 크다는 신호"(§12.5.7)를 실측으로 확인하라.
4. 노름 클리핑을 값 클리핑(`np.clip(g, -c, c)`)으로 바꾸면 (c)의 곡선이 어떻게 달라지는가? 방향 왜곡(§12.5.1)의 효과를 관찰하라.

## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `RHO0` | 1절 | 1.15 | 1.0 이하로 내리면 꼬리 자체가 사라진다 (원인 처방, §12.5.6) |
| `LR` | 1절 | 0.013 | 반으로 내리면 클리핑 없이도 실패가 사라진다 — §12.5.7의 진단 |
| `THETA_STAR` | 4절 | 1.5 | (b)의 두 점선 사이에서 움직여 보라 |
| `B_TR` | 1절 | 16 | 배치가 크면 꼬리가 짧아진다 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")